In [1]:
!pip install --quiet --upgrade pip 
!pip install --quiet --upgrade --no-cache-dir transformers bitsandbytes huggingface_hub
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --no-cache-dir --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.9 MB/s eta 0:00:0000:010:01


In [3]:
import llama_cpp 
print(llama_cpp.llama_supports_gpu_offload())  # Ждем True, проверка на видимость видеокарт библой llama

ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29825 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


True


In [4]:
# Для избежания ограничений по запросам на HF, скачиванию моделей
!export HF_T='ВАШ ТОКЕН'

In [133]:
import gc
import json
import re
import os
import logging
import ast


from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import pandas as pd
import numpy as np
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM,  BitsAndBytesConfig
import bitsandbytes as bnb
import torch

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
df = pd.read_json("hf://datasets/sevenreasons/genius-lyrics-russian/genius-ru.json")
df.head()

,artist,title,featured_artists,tag,tags,lyrics
0,Очередной МС (Another MC),Закономерные случайности (3 раунд 9ob) (Natura...,[],Battle Rap,"[Battle Rap, Русский баттл-рэп (Russian Battle...","[Интро]\nОт создателей ""Красной зелени"", ""Глуб..."
1,Очередной МС (Another MC),Зов природы (4 раунд 9ob) (Call of Nature),[],Русский баттл-рэп (Russian Battle Rap),"[Русский баттл-рэп (Russian Battle Rap), Росси...","[Куплет]\nДети, дети, всё внимание на меня\nРа..."
2,Очередной МС (Another MC),Раунд 5 - Иллюзия свободы (Illyuziya svobody),[],Русский рэп (Russian Rap),"[Русский рэп (Russian Rap), Россия (Russia), Rap]",[Интро]\n(Захочу - вот так сделаю)\n(Я свободе...
3,КИНО (KINO),Видели ночь (Saw the Night),[],СССР (USSR),"[СССР (USSR), Post-Punk, Pop, Русский рок (Rus...",[Текст песни «Видели ночь»]\n\n[Куплет 1]\nМы ...
4,КИНО (KINO),Восьмиклассница (Eighth-Grade Girl),[],СССР (USSR),"[СССР (USSR), Post-Punk, Rock, Русский рок (Ru...",[Текст песни «Восьмиклассница»]\n\n[Куплет 1]\...


In [172]:
prompts = {
    'eng': (
            "Task: Fast extract Russian geographic entities from song lyrics. "
            "Output Format: Strictly JSON array of objects [{'entity': '...', 'type': '...'}]. "
            "Constraints:"
            "1. NO normalization (keep case, gender, number exactly as in text). "
            "2. Hierarchy. Use the smallest possible and only this types: метро < улица < район/округ < город < страна. "
            "3. Response MUST start with '[' and end with ']'. "
            "4. If no Russian toponyms are found, return strictly []."
    ),
    'rus': (
            "Ты — эксперт по извлечению топонимов из русскоязычных текстов песен. "
            "Найди в тексте все именованные географические объекты России. "
            "Верни только JSON-массив объектов формата: [{'entity': 'строка из текста', 'type': 'тип'}]. "
            "Output ONLY a valid JSON list of objects. No intro, no explanations, no thinking. Check your answer on any toponymic homonymy. "
            "Без нормализации, без изменения падежа, числа, регистра и написания. "
            "Возможные типы: улица, метро, район, город, страна, другое."
        )
}

In [197]:
class BaseLLM:
    def __init__(
        self,
        system_prompt=prompts['rus'],
        llma = False
    ):
        self.system_prompt = [{"role": "system", "content": system_prompt}]
        self.model = None
        self.tokenizer = None
        self.model_name = None
        self.model_path = None
        self.max_memory = {0: '13GiB', 1: '13GiB'}  # Ставим для избежания ошибок по памяти, пара Gb под системные нужды
        self.repo_path = 'mudler/Qwen3.5-35B-A3B-APEX-GGUF'
        self.llma = llma
        

    def change_prompt(self, new_prompt):
        self.system_prompt = [{"role": "system", "content": new_prompt}]

    def load(self, model_name='Qwen/Qwen3.5-9B', quant=True):
        self.model_name = model_name

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            

        if not self.llma:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

            
            if quant:
                quant_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_compute_dtype=torch.float16,
                )
    
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config=quant_config,
                    device_map="auto",  # Расщепляем большую модель для запуска на нескольких видеокартах/ставим на одну
                    low_cpu_mem_usage=True,
                    max_memory=self.max_memory
                )
            else:
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    low_cpu_mem_usage=True,
                    max_memory=self.max_memory
                )
                self.model.eval()
        else:
            
            self.model = Llama(
                model_path=hf_hub_download(
                    repo_id = self.repo_path,
                    filename = model_name,
                    token=os.getenv("HF_T")),

                n_gpu_layers=-1,
                n_ctx=4096,
                n_threads=8
            )
        

    def predict(self, text, max_new_tokens=256, temp=0.5):
        messages = self.system_prompt + [{"role": "user", "content": text}]

        if not self.llma:
            inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
            )
    
            inputs = {k: v.to("cuda:0") for k, v in inputs.items()} # не знаю есть ли тут auto, поправить если надо
    
            with torch.no_grad():
                output_tokens = self.model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=max_new_tokens,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
    
            answer = self.tokenizer.decode(
                output_tokens[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True,
            ).strip()
    
            answer = answer.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
            
        else:

            output = self.model.create_chat_completion(
                messages = messages,
                max_tokens=max_new_tokens,
                temperature=temp,
                stop=["<|im_end|>", "<|endoftext|>"]
            )
            answer = output['choices'][0]['message']['content']
            
        return answer

    def predict_json(self, text, max_new_tokens=256, temp=0.5):
        raw = self.predict(text, max_new_tokens=max_new_tokens, temp=temp)
        
        if not self.llma:
            match = re.search(r"\[[\s\S]*\]", raw)
            if not self.llma:
                data = json.loads(match.group(0))
            
        else:
            
            all_matches = re.findall(r"\[[\s\S]*?\]", raw)
            
            if all_matches:
                match = all_matches[-1]
                
                try:
                    data = json.loads(match) or ast.literal_eval(match)
                except Exception as e:
                    print(f'Не смог сохранить json, ошибка: {e}, группа json: {all_matches}, сырой ответ:\n\n{raw}')
            else:
                return []
            
        if not match:
            return []

        try:
            if isinstance(data, list):
                return data
            return []
        except json.JSONDecodeError:
            return []

---
### Классический Qwen без APEX

In [ ]:
# Заводим класс
llm = BaseLLM()
llm.load(model_name='Qwen/Qwen3.5-9B')

In [ ]:
# Если возвращает None - выполнить загрузку модели повторно, может косячить с распределением слоев по видеокартам
print(llm.model)

In [ ]:
%%time
text = "Я снова еду через Тверскую, потом сверну на Арбат и вспомню Москву."
print(llm.predict_json(text, max_new_tokens=1024, temp=0.05))

In [ ]:
llm.change_prompt(prompts['eng'])

In [ ]:
llm.change_prompt(prompts['rus'])

In [ ]:
song = '''
Я рассекаю китай-город
Как мою бровь кулак
Где-то в ссаном переулке на чистых прудах
Патриарших
Там где холод
Пробрал нас до дрожи
Мы едва ли живы
На остоженке лежим
@
Мы бежим через садовое
Забыв о светофорах
Направляемся к савёле
За закладкой под забором
Мы взрываем у высотки
На красных воротах
Мы целуемся взасос
В зоопарке как животные
@
В парке дружбы на речном
Ебемся прямо на газоне
Мы смеёмся над позерами
На цветном бульваре
Молодые мародеры
@
Мы воруем в ресторанах
Маски-шоу — мы дали дёру
С рейва вовремя удрали
Ты пиздатая девчонка
А я охуенный парень
'''

In [ ]:
%%time
for i in song.split('@'):
    print(llm.predict_json(i, max_new_tokens=1024, temp=0.1))

---

## Тренируем QWEN на архитектуре APEX

In [198]:
# Заводим класс
llm = BaseLLM(llma=True)

In [201]:
# Лучшая, но долго думает - Qwen3.5-35B-A3B-APEX-I-Quality.gguf
# Самая мелкая, но качественная среди всех - Qwen3.5-35B-A3B-APEX-I-Compact.gguf
llm.load(model_name='Qwen3.5-35B-A3B-APEX-I-Quality.gguf')

llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) (0000:00:04.0) - 14789 MiB free
llama_model_load_from_file_impl: using device CUDA1 (Tesla T4) (0000:00:05.0) - 14789 MiB free
llama_model_loader: loaded meta data with 50 key-value pairs and 733 tensors from /root/.cache/huggingface/hub/models--mudler--Qwen3.5-35B-A3B-APEX-GGUF/snapshots/917b86edb115c0deac65d1269b7aae6e274ddf61/Qwen3.5-35B-A3B-APEX-I-Quality.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen35moe
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                    

In [202]:
# Если возвращает None - выполнить загрузку модели повторно, может косячить с распределением слоев по видеокартам
print(llm.model)

In [203]:
%%time
text = "Я снова еду через Тверскую, потом сверну на Арбат и вспомню Москву."
print(llm.predict_json(text, max_new_tokens=1024, temp=0.05))


ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     512.16 ms
llama_perf_context_print: prompt eval time =     511.75 ms /   161 tokens (    3.18 ms per token,   314.61 tokens per second)
llama_perf_context_print:        eval time =   18466.39 ms /  1023 runs   (   18.05 ms per token,    55.40 tokens per second)
llama_perf_context_print:       total time =   20589.14 ms /  1184 tokens
llama_perf_context_print:    graphs reused =       1018


[{'entity': 'Тверскую', 'type': 'улица'}, {'entity': 'Арбат', 'type': 'улица'}, {'entity': 'Москву', 'type': 'город'}]
CPU times: user 20.5 s, sys: 149 ms, total: 20.6 s
Wall time: 20.6 s


In [ ]:
llm.change_prompt(prompts['eng'])

In [181]:
llm.change_prompt(prompts['rus'])

In [204]:
song = '''
Я рассекаю китай-город
Как мою бровь кулак
Где-то в ссаном переулке на чистых прудах
Патриарших
Там где холод
Пробрал нас до дрожи
Мы едва ли живы
На остоженке лежим
@
Мы бежим через садовое
Забыв о светофорах
Направляемся к савёле
За закладкой под забором
Мы взрываем у высотки
На красных воротах
Мы целуемся взасос
В зоопарке как животные
@
В парке дружбы на речном
Ебемся прямо на газоне
Мы смеёмся над позерами
На цветном бульваре
Молодые мародеры
@
Мы воруем в ресторанах
Маски-шоу — мы дали дёру
С рейва вовремя удрали
Ты пиздатая девчонка
А я охуенный парень
'''

In [205]:
%%time
for i in song.split('@'):
    print(llm.predict_json(i, max_new_tokens=1024, temp=0.1))

Llama.generate: 137 prefix-match found but partial kv removal not supported, re-evaluating full prompt
llama_perf_context_print:        load time =     512.16 ms
llama_perf_context_print: prompt eval time =     589.50 ms /   209 tokens (    2.82 ms per token,   354.54 tokens per second)
llama_perf_context_print:        eval time =   18384.88 ms /  1023 runs   (   17.97 ms per token,    55.64 tokens per second)
llama_perf_context_print:       total time =   20565.41 ms /  1232 tokens
llama_perf_context_print:    graphs reused =       1018


Не смог сохранить json, ошибка: Expecting property name enclosed in double quotes: line 1 column 3 (char 2), группа json: ["[{'entity': 'string', 'type': 'type'}]"], сырой ответ:

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Extract named geographical objects (toponyms) from the provided Russian song lyrics.
    *   Target: Objects located in Russia.
    *   Output Format: JSON array of objects `[{'entity': 'string', 'type': 'type'}]`.
    *   Constraints:
        *   Only valid JSON.
        *   No intro, no explanations, no thinking.
        *   Check for toponymic homonymy (ensure they are actually places in Russia).
        *   No normalization (keep original case, declension, spelling).
        *   Possible types: `улица` (street), `метро` (metro), `район` (district), `город` (city), `страна` (country), `другое` (other).

2.  **Analyze the Input Text:**
    ```
    Я рассекаю китай-город
    Как мою бровь кулак
    Где-то в ссаном переулке на чистых прудах
    Пат

UnboundLocalError: cannot access local variable 'data' where it is not associated with a value

In [206]:
llm.change_prompt(prompts['eng'])

In [207]:
%%time
for i in song.split('@'):
    print(llm.predict_json(i, max_new_tokens=1024, temp=0.05))

Llama.generate: 3 prefix-match found but partial kv removal not supported, re-evaluating full prompt
llama_perf_context_print:        load time =     512.16 ms
llama_perf_context_print: prompt eval time =     556.17 ms /   187 tokens (    2.97 ms per token,   336.23 tokens per second)
llama_perf_context_print:        eval time =   18455.24 ms /  1023 runs   (   18.04 ms per token,    55.43 tokens per second)
llama_perf_context_print:       total time =   20619.94 ms /  1210 tokens
llama_perf_context_print:    graphs reused =       1018
Llama.generate: 114 prefix-match found but partial kv removal not supported, re-evaluating full prompt


[]


llama_perf_context_print:        load time =     512.16 ms
llama_perf_context_print: prompt eval time =     429.82 ms /   183 tokens (    2.35 ms per token,   425.75 tokens per second)
llama_perf_context_print:        eval time =   18881.07 ms /  1023 runs   (   18.46 ms per token,    54.18 tokens per second)
llama_perf_context_print:       total time =   20911.13 ms /  1206 tokens
llama_perf_context_print:    graphs reused =       1018
Llama.generate: 114 prefix-match found but partial kv removal not supported, re-evaluating full prompt


[]


llama_perf_context_print:        load time =     512.16 ms
llama_perf_context_print: prompt eval time =     392.99 ms /   161 tokens (    2.44 ms per token,   409.68 tokens per second)
llama_perf_context_print:        eval time =   18295.01 ms /  1023 runs   (   17.88 ms per token,    55.92 tokens per second)
llama_perf_context_print:       total time =   20276.59 ms /  1184 tokens
llama_perf_context_print:    graphs reused =       1018


Не смог сохранить json, ошибка: Expecting value: line 1 column 2 (char 1), группа json: ["[{'entity': '...', 'type': '...'}]", "[' and end with ']", '[]', '[side?]', '[side]'], сырой ответ:

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Extract Russian geographic entities from song lyrics.
    *   Input: A snippet of song lyrics ("В парке дружбы на речном / Ебемся прямо на газоне / Мы смеёмся над позерами / На цветном бульваре / Молодые мародеры").
    *   Output Format: Strictly JSON array of objects `[{'entity': '...', 'type': '...'}]`.
    *   Constraints:
        1.  NO normalization (keep case, gender, number exactly as in text).
        2.  Hierarchy: Use the smallest possible type: метро < улица < район/округ < город < страна.
        3.  Response MUST start with '[' and end with ']'.
        4.  If no Russian toponyms are found, return strictly `[]`.

2.  **Analyze the Input Text:**
    *   "В парке дружбы на речном" (In Friendship Park on the river [side?])
   

UnboundLocalError: cannot access local variable 'data' where it is not associated with a value

In [ ]:
%%time

song = '''
(Куплет 1)
Вылетаю с Васьки к Василию, пока мосты не свели,
В кармане ноль, но у Павла в Думе огни.
Ныряю в Гостинку, к Константину на шум,
Этот город, как Шнуров, забирает мой ум.
На Лиговке Лиза мутит какой-то варик,
У Галереи Георгий зажег свой фонарик.
Мимо Грибанала, где Грибоедов и лёд,
Петр Первый меня манит, Петр Первый меня ждёт.

(Припев)
О, этот город Достоевского и пустых обещаний,
Где каждый Родион хранит сотни прощаний.
От Петроградки до гетто в Купчино,
Где Данила и Брат смотрят в окно.

(Куплет 2)
Занесло на Рубика, там Рубинштейн и блеск,
У Белинского Виссарион слушает плеск.
На Чёрной речке Александр Сергеич упал,
А в Мурино Артем свой бетонометр вкопал.
Сверну на Можайку, к Александру в подвал,
Коломна — как Блок, что аптеку искал.
На Техноложке Дмитрий Иванович курит в гул,
В Пышечной у Надежды я в сахар нырнул.

(Бридж)
Никаких поребриков, только Борис и хардкор,
С Просвета Анатолий затевает спор.
Девяткино плачет, как Ксения вдали,
Мы в Лахте у Никиты приют обрели.

(Куплет 3)
Пора на Невский, там Николай начал выползать,
Где Казань и Мария готовы подпевать.
Но ноги несут, где Севкабель и Сева гудит,
Где Порт и Иосиф нам свободу сулит.
Зависну на Садовой, где Виктор и злой,
Мимо Сенной, где Федор стал сам собой.
На Пяти углах Сергей сводит пути,
Отсюда, Андрей, уже не уйти.

(Финал)
Серый Питер.
Васька, Петроградка, Лиговка...
Большевиков, Академка, Парнас.
Ксения, Павел, Степан и все нас.
Город-человек.
'''

llm.predict(song)

In [158]:
song = df['lyrics'][df['lyrics'].str.contains('ЮВАО')].iloc[14]
print(song)


[Текст песни «Angry toy$»]

[Припев]
Прихожу раньше, не то чтобы вовремя
Покажу пальцем, и ты станешь донором
Не звоню копам — не знаю их номера
Злые намерения бьют в мою голову

[Куплет]
Возьму себя в руки и все свои «но»
Да мне хоть Купчино, хоть ЮВАО
С миром хожу, он со мной заодно
На мне заклятье на полный armor
С тех самых пор, как я вышел на pro-режим
Автоматически стал им всем корешем
Я безразличен к тому, что ты порешь
И нахуй ты нужен? Ты нахуй звонишь?
How do you feel? Йоу, how do you feel?
Похуй на ксивы и что там за папа
Высока стата — мне много не надо
Живу эту жизнь, и мне нравится мапа
Подмена понятий — суровая правда
Трудно, knock'абельно, я понимаю
Никаких чувств, ты застыл, будто камень
В странной одежде, в своей географии
Мой money talk — колдовство над деньгами
Иду по Арбату, и все меня палят
Жизнь на виду меня искренне калит

[Припев]
Прихожу раньше, не то чтобы вовремя
Покажу пальцем, и ты станешь донором
Не звоню копам — не знаю их номера
Злые намерения бьют в мо

In [169]:
%%time
for i in song.split(']'):
    print(llm.predict_json(i, max_new_tokens=1024, temp=0.1))

Llama.generate: 58 prefix-match found but partial kv removal not supported, re-evaluating full prompt
llama_perf_context_print:        load time =     655.90 ms
llama_perf_context_print: prompt eval time =     523.26 ms /   138 tokens (    3.79 ms per token,   263.73 tokens per second)
llama_perf_context_print:        eval time =     283.38 ms /    16 runs   (   17.71 ms per token,    56.46 tokens per second)
llama_perf_context_print:       total time =    1543.09 ms /   154 tokens
llama_perf_context_print:    graphs reused =         15
Llama.generate: 121 prefix-match found but partial kv removal not supported, re-evaluating full prompt


[{'entity': 'Angry toy$', 'type': 'song'}]


llama_perf_context_print:        load time =     655.90 ms
llama_perf_context_print: prompt eval time =     352.55 ms /   130 tokens (    2.71 ms per token,   368.74 tokens per second)
llama_perf_context_print:        eval time =     246.10 ms /    15 runs   (   16.41 ms per token,    60.95 tokens per second)
llama_perf_context_print:       total time =    1277.49 ms /   145 tokens
llama_perf_context_print:    graphs reused =         14
Llama.generate: 120 prefix-match found but partial kv removal not supported, re-evaluating full prompt


[{'entity': 'Припев', 'type': 'district'}]


llama_perf_context_print:        load time =     655.90 ms
llama_perf_context_print: prompt eval time =     459.31 ms /   180 tokens (    2.55 ms per token,   391.90 tokens per second)
llama_perf_context_print:        eval time =     264.29 ms /    16 runs   (   16.52 ms per token,    60.54 tokens per second)
llama_perf_context_print:       total time =    1450.18 ms /   196 tokens
llama_perf_context_print:    graphs reused =         15
Llama.generate: 120 prefix-match found but partial kv removal not supported, re-evaluating full prompt


[{'entity': 'России', 'type': 'region'}]


llama_perf_context_print:        load time =     655.90 ms
llama_perf_context_print: prompt eval time =     720.01 ms /   371 tokens (    1.94 ms per token,   515.27 tokens per second)
llama_perf_context_print:        eval time =     936.49 ms /    44 runs   (   21.28 ms per token,    46.98 tokens per second)
llama_perf_context_print:       total time =    3639.39 ms /   415 tokens
llama_perf_context_print:    graphs reused =         43
Llama.generate: 120 prefix-match found but partial kv removal not supported, re-evaluating full prompt


[{'entity': 'Купчино', 'type': 'district'}, {'entity': 'ЮВАО', 'type': 'district'}, {'entity': 'Арбат', 'type': 'street'}]


llama_perf_context_print:        load time =     655.90 ms
llama_perf_context_print: prompt eval time =     651.03 ms /   175 tokens (    3.72 ms per token,   268.80 tokens per second)
llama_perf_context_print:        eval time =     277.70 ms /    16 runs   (   17.36 ms per token,    57.62 tokens per second)
llama_perf_context_print:       total time =    1669.92 ms /   191 tokens
llama_perf_context_print:    graphs reused =         15


[{'entity': 'России', 'type': 'region'}]
CPU times: user 8.89 s, sys: 764 ms, total: 9.65 s
Wall time: 9.63 s


In [ ]:
%%time
answer = llm.predict(song)
print(answer)

In [ ]:
song = df['lyrics'][df['lyrics'].str.contains('Москва')].iloc[6]
print(song)

In [ ]:
%%time
answer = llm.predict(song)
print(answer)

In [ ]:
song = df['lyrics'][df['lyrics'].str.contains('Питер')].iloc[5]
print(song)

In [ ]:
%%time
answer = llm.predict(song)
print(answer)

In [ ]:
song = df['lyrics'][df['lyrics'].str.contains('Китай-город')].iloc[9]
print(song)

In [ ]:
%%time
answer = llm.predict(song)
print(answer)

In [ ]:
song = df['lyrics'][df['lyrics'].str.contains('Москва')].iloc[2]
print(song)

In [ ]:
%%time
answer = llm.predict(song)
print(answer)